# 01 — Data Wrangling

This notebook loads the balanced analytical sample for the YouTube trending project, validates the 2024–2025 balance, cleans data types, and engineers the features used later in EDA and modeling.

Note: in this environment, the available file is the balanced sample `youtube_2024_2025_balanced_sample.csv` (100,000 rows from 2024 and 100,000 rows from 2025). It was created from the original Kaggle dataset in 00_create_2024_2025_sample.ipynb notebook.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
BASE_DIR = Path(".")
INPUT_FILE = BASE_DIR / "youtube_2024_2025_balanced_sample.csv"
ROW_OUTPUT = BASE_DIR / "yt_project_outputs" / "analysis_row_level.csv"
VIDEO_OUTPUT = BASE_DIR / "yt_project_outputs" / "analysis_video_level.csv"

INPUT_FILE.exists(), INPUT_FILE

(True, PosixPath('youtube_2024_2025_balanced_sample.csv'))

In [2]:
df = pd.read_csv(INPUT_FILE, low_memory=False, parse_dates=["snapshot_date", "publish_date"])
df = df.rename(columns={"langauge": "language"})
print(df.shape)
df.head()

(200000, 20)


,title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,like_count,comment_count,description,thumbnail_url,video_id,channel_id,video_tags,kind,publish_date,language,year,month
0,DIY cardboard slide for Kids #ahorts #diy #par...,5-Minute Crafts LIKE,44,-22,6,2024-04-06,MK,13422404,261526,308,NaN,https://i.ytimg.com/vi/Euv1Pc2SLXc/mqdefault.jpg,Euv1Pc2SLXc,UCzTWHWnJ6VIcbkpqyv7_FaQ,NaN,youtube#video,2024-03-21 00:00:00+00:00,en,2024,4
1,TUNDU LISSU ASHINDWA KUVUMILIA MSIBANI KWA LOW...,Bongo5,19,-4,31,2024-02-17,TZ,176865,580,109,Bongo5 is a dedicated to creating the best con...,https://i.ytimg.com/vi/jk88W3CsRfA/mqdefault.jpg,jk88W3CsRfA,UCrJ5CGMLyFFLnKtVNpgxtrQ,"bongo5updates, Rais Samia, BBC Swahili, Millar...",youtube#video,2024-02-12 00:00:00+00:00,en,2024,2
2,JERE KLEIN - OTRA VOLÁ FEAT SANTAFERIA (VIDEO ...,Jere Klein,35,-4,-14,2024-12-28,CL,2167082,0,1569,JERE KLEIN PRESENTA SU NUEVO SINGLE OTRA VOLÁ ...,https://i.ytimg.com/vi/eMTQz-TpHC8/mqdefault.jpg,eMTQz-TpHC8,UCcJj2mZoB07p1IxHtXZU7bg,"jere klein, jere klein santaferia, jere klein ...",youtube#video,2024-12-19 00:00:00+00:00,es-419,2024,12
3,Become a Diamond Millionaire in Hay Day,Hay Day,22,13,-20,2024-06-18,BH,1803863,45297,1332,"Best. Farm. Ever. Welcome to Hay Day, the most...",https://i.ytimg.com/vi/E9d6h0dQ4qY/mqdefault.jpg,E9d6h0dQ4qY,UCBrMq7wkdPWb3PxR-GUR61A,"Hay Day, HayDay, Supercell, Suprcell, Farming,...",youtube#video,2024-06-11 00:00:00+00:00,en,2024,6
4,شاهد الرسالة التي وجهها أنصار الخضر للناخب بلم...,ELBILAD TV,7,43,43,2024-01-25,TN,349244,4419,2621,شاهد الرسالة التي وجهها أنصار #المنتخب_الوطني ...,https://i.ytimg.com/vi/jVzxv1PI4So/mqdefault.jpg,jVzxv1PI4So,UCfF6CkUvCBleM19CkgNyrYw,"البلاد, قناة البلاد, البلاد tv, البلاد تي في, ...",youtube#video,2024-01-23 00:00:00+00:00,NaN,2024,1


In [3]:
# Basic schema review
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_pct": [df[c].isna().mean().round(4) for c in df.columns],
})
schema

,column,dtype,missing_pct
0,title,object,0.0000
1,channel_name,object,0.0000
2,daily_rank,int64,0.0000
3,daily_movement,int64,0.0000
4,weekly_movement,int64,0.0000
5,snapshot_date,datetime64[ns],0.0000
6,country,object,0.0000
7,view_count,int64,0.0000
8,like_count,int64,0.0000
9,comment_count,int64,0.0000


In [4]:
# Clean and engineer core features
df["description"] = df["description"].fillna("")
df["video_tags"] = df["video_tags"].fillna("")
df["title"] = df["title"].fillna("")
df["language"] = df["language"].fillna("unknown")
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], errors="coerce")
df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True).dt.tz_convert(None)

df["days_to_trend"] = (df["snapshot_date"] - df["publish_date"]).dt.total_seconds() / 86400
df["days_to_trend"] = df["days_to_trend"].clip(lower=0)

df["likes_to_views_ratio"] = np.where(df["view_count"] > 0, df["like_count"] / df["view_count"], np.nan)
df["comments_to_views_ratio"] = np.where(df["view_count"] > 0, df["comment_count"] / df["view_count"], np.nan)

df["description_length"] = df["description"].str.len()
df["title_length"] = df["title"].str.len()
df["uppercase_ratio"] = df["title"].apply(lambda s: sum(ch.isupper() for ch in s) / max(sum(ch.isalpha() for ch in s), 1))
df["exclamation_count"] = df["title"].str.count("!")
df["question_count"] = df["title"].str.count(r"\?")
df["publish_weekday"] = df["publish_date"].dt.day_name()
df["publish_hour_utc"] = df["publish_date"].dt.hour
df["publish_month"] = df["publish_date"].dt.month
df["hashtag_count"] = df["title"].str.count("#") + df["description"].str.count("#")
df["tag_count"] = np.where(df["video_tags"].eq(""), 0, df["video_tags"].str.count(",") + 1)
df["language_clean"] = (
    df["language"].astype(str).str.lower().str.split("-").str[0].replace({"und": "unknown", "nan": "unknown"})
)

df[["days_to_trend", "likes_to_views_ratio", "description_length", "title_length"]].describe().round(3)

,days_to_trend,likes_to_views_ratio,description_length,title_length
count,200000.000,199957.000,200000.000,200000.000
mean,6.465,0.040,715.513,51.092
std,5.561,0.033,826.699,23.424
min,0.000,0.000,0.000,1.000
25%,2.000,0.017,136.000,33.000
50%,5.000,0.031,463.000,47.000
75%,9.000,0.053,973.000,67.000
max,37.000,1.003,5000.000,100.000


In [5]:
# Validate sample balance
year_balance = df["year"].value_counts().sort_index()
month_balance = df.groupby(["year", "month"]).size().unstack(fill_value=0)
country_balance = df.groupby(["year", "country"]).size()

print("Rows by year")
display(year_balance)

print("\nMonthly balance by year")
display(month_balance)

print("\nCountry balance summary by year")
country_balance.groupby(level=0).agg(["min", "max", "mean", "std"]).round(2)

Rows by year


year
2024    100000
2025    100000
Name: count, dtype: int64


Monthly balance by year


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2024,8589,8038,8201,8270,8425,8291,8260,8421,8252,8511,8172,8570
2025,8592,7823,8705,8255,8582,8274,8473,8282,8054,8465,8049,8446



Country balance summary by year


,min,max,mean,std
year,,,,
2024,832,950,884.96,25.03
2025,617,962,884.96,46.16


In [6]:
# Build a video-level table for questions that should not overweight repeated appearances
video_level = (
    df.groupby("video_id")
      .agg(
          title=("title", "first"),
          channel_name=("channel_name", "first"),
          first_snapshot_date=("snapshot_date", "min"),
          publish_date=("publish_date", "first"),
          country_span=("country", "nunique"),
          sample_rows=("video_id", "size"),
          avg_view_count=("view_count", "mean"),
          max_view_count=("view_count", "max"),
          avg_like_count=("like_count", "mean"),
          avg_comment_count=("comment_count", "mean"),
          avg_engagement_rate=("likes_to_views_ratio", "mean"),
          median_engagement_rate=("likes_to_views_ratio", "median"),
          description=("description", "first"),
          language_clean=("language_clean", lambda x: x.mode().iat[0] if not x.mode().empty else "unknown"),
          country_first=("country", "first"),
          publish_hour_utc=("publish_hour_utc", "first"),
          year=("year", "first")
      )
      .reset_index()
)

video_level["days_to_first_trend"] = (
    video_level["first_snapshot_date"] - video_level["publish_date"]
).dt.total_seconds() / 86400
video_level["days_to_first_trend"] = video_level["days_to_first_trend"].clip(lower=0)
video_level["publish_weekday"] = video_level["publish_date"].dt.day_name()
video_level["description_length"] = video_level["description"].fillna("").str.len()
video_level["title_length"] = video_level["title"].fillna("").str.len()

print(df.shape, video_level.shape)
video_level.head()

(200000, 34) (91033, 22)


,video_id,title,channel_name,first_snapshot_date,publish_date,country_span,sample_rows,avg_view_count,max_view_count,avg_like_count,avg_comment_count,avg_engagement_rate,median_engagement_rate,description,language_clean,country_first,publish_hour_utc,year,days_to_first_trend,publish_weekday,description_length,title_length
0,--2DpWkbo2U,Spektakulärer Umbau: junge Familie baut alte S...,ARD Room Tour,2024-09-02,2024-09-01,1,1,60746.0,60746,1297.0,110.0,0.021351,0.021351,"Rot, grün, blau, lila, gelb – Erika, Willi und...",de,DE,0,2024,1.0,Sunday,4550,91
1,--2k07O8VaY,Radio Mileva | Epizoda 21 (domaća serija),RTS TV serije - Zvanični kanal,2024-08-23,2024-08-15,1,1,185679.0,185679,2813.0,114.0,0.015150,0.015150,"Mileva, penzionisana sekretarica koja je radil...",sr,RS,0,2024,8.0,Thursday,1572,41
2,--4JJPg6XD0,UNBOXING HADIAH dari MasHazard untuk Anak-anak...,MasHazard,2024-02-16,2024-02-14,1,1,117047.0,117047,7246.0,282.0,0.061907,0.061907,"Hesebah Halo Guys, jadi pada episode 3 kali in...",en,ID,0,2024,2.0,Wednesday,217,64
3,--4o4X_JqwU,Erick es infiel. Broma pesada a Yess | Carolin...,Carolina Díaz,2024-05-14,2024-05-13,5,5,643963.2,948271,32446.6,690.0,0.053042,0.049611,"Hoy traemos VENGANZA, una venganza que me habí...",es,CO,0,2024,1.0,Monday,356,52
4,--5SKkPzeM4,РЕТРОГРАДНЫЙ МЕРКУРИЙ с 26 ноября по 15 декабр...,Angela Pearl,2024-11-26,2024-11-25,1,1,421272.0,421272,26935.0,1326.0,0.063937,0.063937,И снова пришло время Ретро Меркурия. На этот р...,ru,RU,0,2024,1.0,Monday,325,97


In [7]:
ROW_OUTPUT.parent.mkdir(exist_ok=True)
df.to_csv(ROW_OUTPUT, index=False)
video_level.to_csv(VIDEO_OUTPUT, index=False)

print(f"Saved row-level analysis file to: {ROW_OUTPUT}")
print(f"Saved video-level analysis file to: {VIDEO_OUTPUT}")

Saved row-level analysis file to: yt_project_outputs/analysis_row_level.csv
Saved video-level analysis file to: yt_project_outputs/analysis_video_level.csv


## Summary

The sample is balanced at 100,000 rows per year across 2024 and 2025, while still covering 113 countries. The cleaned row-level and video-level files are now ready for EDA and modeling.